In [ ]:
#@title Install
!pip install -q pandas numpy

In [ ]:
#@title Load runs
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, re, math, statistics as st
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 200)

ROOT = Path('/content/drive/MyDrive/bearing_protocol')
RUNS_DIR=ROOT/'runs'; EVAL_DIR=ROOT/'eval'; CACHE=EVAL_DIR/'judge_cache'
OUT=ROOT/'tables'; OUT.mkdir(parents=True, exist_ok=True)
DATA_DIRS={'PB':ROOT/'data_pb_5khz','CW':ROOT/'data_cw_5khz'}
SPEC={'PB':'spec_v4.json','CW':'spec_cw_v2.json'}; MANIFEST={'PB':'data_manifest_v4.json','CW':'data_manifest_cw_v2.json'}
JUDGE_TAG='Qwen2.5-32B-Instruct'
CONDS=['correct','contrastive','blind','substituted']
CLASSES=['Healthy','Inner_Ring_Fault','Outer_Ring_Fault']
QN=['env_char_energy_frac','env_bpfo_ratio','env_bpfo_harmonic_ratio','env_bpfi_ratio','env_bpfi_sideband_ratio']
QSHORT={'env_char_energy_frac':'char. energy','env_bpfo_ratio':'BPFO','env_bpfo_harmonic_ratio':'BPFO harm.',
        'env_bpfi_ratio':'BPFI','env_bpfi_sideband_ratio':'BPFI sideband'}
PARSER_MODELS={'qwen3omni-30b','qwen25omni-7b'}

DS=[ds for ds in DATA_DIRS if (DATA_DIRS[ds]/SPEC[ds]).exists()]
TH={ds:json.load(open(DATA_DIRS[ds]/SPEC[ds]))['thresholds'] for ds in DS}
def true_level(q,v,ds): lo,hi=TH[ds][q]; return 'low' if v<lo else ('high' if v>=hi else 'mid')
MODE={}
for ds in DS:
    man=json.load(open(DATA_DIRS[ds]/MANIFEST[ds]))
    MODE[ds]={c:{q:Counter(true_level(q,r['features'][q],ds) for r in man['train'] if r['label']==c).most_common(1)[0][0]
                 for q in QN} for c in CLASSES}

RUNS=[]
for p in sorted(RUNS_DIR.glob('*.json')):
    r=json.load(open(p))
    if r.get('status')!='done' or not r.get('result_dir') or not r.get('subsets') or r.get('dataset') not in DS: continue
    d=Path(r['result_dir']); conds=[c for c in CONDS if (d/f'infer_{c}.jsonl').exists()]
    if conds:
        RUNS.append(dict(rid=r['run_id'],dir=d,conds=conds,tag=r['model'],label=r['model'],dataset=r['dataset'],
                         modality=r['modality'],ratio=int(r['ratio']),seed=int(r['seed']),subsets=r['subsets']))
RUN={r['rid']:r for r in RUNS}

P2C={'healthy bearing':'Healthy','inner ring fault':'Inner_Ring_Fault','outer ring fault':'Outer_Ring_Fault'}
PATS={'env_char_energy_frac':[(r'only a small fraction of the envelope energy','low'),(r'a moderate fraction of the envelope energy','mid'),(r'a large fraction of the envelope energy','high')],
      'env_bpfo_ratio':[(r'little energy at the outer-race','low'),(r'moderate energy at the outer-race','mid'),(r'strongly concentrated at the outer-race','high')],
      'env_bpfo_harmonic_ratio':[(r'no harmonic banding','low'),(r'weak harmonic content','mid'),(r'clear harmonic banding','high')],
      'env_bpfi_ratio':[(r'little energy at the inner-race','low'),(r'moderate energy at the inner-race','mid'),(r'strongly concentrated at the inner-race','high')],
      'env_bpfi_sideband_ratio':[(r'no shaft-rate sideband','low'),(r'weak shaft-rate sideband','mid'),(r'clear shaft-rate sideband','high')]}
def parse(text):
    t=re.sub(r'<think>.*?</think>',' ',text or '',flags=re.S|re.I)
    b=re.search(r'<explanation>(.*?)</explanation>', t, re.S|re.I); low=(b.group(1) if b else t).lower()
    m=re.search(r'consistent with (?:a|an)\s+(healthy bearing|inner ring fault|outer ring fault)', low)
    return {'class':P2C[m.group(1)] if m else None,
            'rejected':bool(re.search(r'however,\s*the cnn classified this as', low)),
            'claims':{q:next((l for p,l in pats if re.search(p,low)),None) for q,pats in PATS.items()}}
def resp(g):
    for k in ('response','generated_text','completion'):
        if isinstance(g.get(k),str) and g[k]: return g[k]
    m=g.get('messages'); return m[-1].get('content','') if m and m[-1].get('role')=='assistant' else ''
def load_jsonl(p):
    with open(p) as f: return [json.loads(l) for l in f if l.strip()]
DATA={}; JUD={}
for r in RUNS:
    for cond in r['conds']:
        sub=Path(r['subsets']['blind' if cond=='substituted' else cond])
        meta=load_jsonl(sub); gen=load_jsonl(r['dir']/f'infer_{cond}.jsonl')
        if len(meta)!=len(gen): print(f"  ! {r['rid']}/{cond}: {len(meta)} vs {len(gen)}"); continue
        DATA[(r['rid'],cond)]=[{**m['meta'],**parse(resp(g))} for m,g in zip(meta,gen)]
        cp=CACHE/f"{r['rid']}__{cond}__{JUDGE_TAG}.json"
        if cp.exists(): JUD[(r['rid'],cond)]={int(k):v for k,v in json.loads(cp.read_text()).items()}
print(f'{len(RUNS)} runs; {len(DATA)} run/condition pairs with generations; {len(JUD)} with 32B readings')

In [ ]:
#@title Metrics
N_BOOT=2000
def view(rid, cond, reader):
    rows=DATA.get((rid,cond))
    if rows is None: return None
    if reader=='parser':
        return [dict(cls=x['class'],rej=x['rejected'],claims=x['claims']) for x in rows]
    J=JUD.get((rid,cond))
    if J is None: return None
    norm=lambda v: v if v in ('low','mid','high') else None
    return [dict(cls=(J.get(i,{}).get('class') if J.get(i,{}).get('class') in CLASSES else None),
                 rej=(J.get(i,{}).get('verdict')=='DISAGREE'),
                 claims={q:norm(J.get(i,{}).get(q)) for q in QN}) for i in range(len(rows))]

def boot_mean(v, rng):
    v=np.asarray(v,float); B=rng.integers(0,len(v),(N_BOOT,len(v))); return v[B].mean(1)
def ci_of(samples): return [round(float(100*np.percentile(samples,2.5)),2), round(float(100*np.percentile(samples,97.5)),2)]

def metrics(rid, reader):
    r=RUN[rid]; ds=r['dataset']; out={'reader':reader}; rng=np.random.default_rng(0)
    vc,vx=view(rid,'correct',reader),view(rid,'contrastive',reader)
    if vc and vx:
        dc,dx=DATA[(rid,'correct')],DATA[(rid,'contrastive')]
        a=np.array([v['cls']==d['supplied'] for v,d in zip(vc,dc)],float)
        b=np.array([v['cls']==d['label'] for v,d in zip(vx,dx)],float)
        rj=np.array([v['rej'] for v in vx],float); echo=np.array([v['cls']==d['supplied'] for v,d in zip(vx,dx)],float)
        sa,sb=boot_mean(a,rng),boot_mean(b,rng)
        out.update(confirm=100*a.mean(), named=100*b.mean(), reject=100*rj.mean(), echo=100*echo.mean(),
                   arb=100*(a.mean()+b.mean())/2, arb_ci=ci_of((sa+sb)/2))
    for cond in CONDS:
        v=view(rid,cond,reader)
        if v is None: continue
        d=DATA[(rid,cond)]
        hit=np.array([[x['claims'][q]==true_level(q,y['features'][q],ds) for q in QN] for x,y in zip(v,d)],float)
        stated=np.array([[x['claims'][q] is not None for q in QN] for x in v],float)
        hs=hit.mean(1); B=rng.integers(0,len(hs),(N_BOOT,len(hs)))
        e=dict(acc=100*hit.mean(), acc_ci=ci_of(hs[B].mean(1)), coverage=100*stated.mean(),
               acc_made=(100*hit.sum()/stated.sum() if stated.sum() else float('nan')),
               per_q={q:100*hit[:,j].mean() for j,q in enumerate(QN)})
        nt=np.array([[MODE[ds][y['label']][q]==true_level(q,y['features'][q],ds) for q in QN] for y in d],float)
        nts=nt.mean(1)
        e.update(null_true=100*nt.mean(), margin_true=100*(hit.mean()-nt.mean()), margin_true_ci=ci_of((hs-nts)[B].mean(1)))
        chosen, which = nt, 'true'
        if cond in ('correct','contrastive'):
            nul=np.array([[x['cls'] in MODE[ds] and MODE[ds][x['cls']][q]==true_level(q,y['features'][q],ds) for q in QN]
                          for x,y in zip(v,d)],float)
            ns=nul.mean(1)
            e.update(null_concl=100*nul.mean(), margin_concl=100*(hit.mean()-nul.mean()), margin_concl_ci=ci_of((hs-ns)[B].mean(1)))
            if nul.mean()>nt.mean(): chosen, which = nul, 'concluded'
        cs=chosen.mean(1)
        e.update(null=100*chosen.mean(), null_from=which, margin=100*(hit.mean()-chosen.mean()), margin_ci=ci_of((hs-cs)[B].mean(1)))
        if stated.sum():
            e['margin_made']=100*(hit.sum()-(chosen*stated).sum())/stated.sum()
        out[cond]=e
    return out

M={}
for r in RUNS:
    M[r['rid']]={rd:metrics(r['rid'],rd) for rd in ('parser','32b')}
def primary(rid): return 'parser' if RUN[rid]['tag'] in PARSER_MODELS else '32b'
def P(rid): return M[rid][primary(rid)]
print(f'metrics for {len(M)} runs, both readers')

In [ ]:
#@title Every run
f1=lambda v: (f'{v:.1f}' if isinstance(v,(int,float)) and v==v else '–')
def fci(v,c): return f'{f1(v)} [{f1(c[0])}, {f1(c[1])}]' if c else f1(v)
def g(m,*path):
    for k in path:
        if not isinstance(m,dict) or k not in m: return float('nan')
        m=m[k]
    return m
rows=[]
for r in sorted(RUNS, key=lambda r:(r['dataset'],r['tag'],r['modality'],r['ratio'],r['seed'])):
    p=P(r['rid']); o=M[r['rid']]['32b' if primary(r['rid'])=='parser' else 'parser']
    row={'dataset':r['dataset'],'model':r['tag'],'input':r['modality'],'rho':r['ratio']/100,'seed':r['seed'],
         'reader':primary(r['rid']),
         'arb BA [95% CI]':fci(p.get('arb'),p.get('arb_ci')),'confirms correct':f1(p.get('confirm')),
         'rejects incorrect':f1(p.get('reject')),'names true':f1(p.get('named')),'echoes incorrect':f1(p.get('echo'))}
    for cond in CONDS:
        if cond in p:
            row[f'evid {cond} [95% CI]']=fci(g(p,cond,'acc'),g(p,cond,'acc_ci'))
    for cond in ('correct','contrastive'):
        if cond in p:
            row[f'null {cond}']=f"{f1(g(p,cond,'null'))} ({g(p,cond,'null_from')})"
            row[f'margin {cond} [95% CI]']=fci(g(p,cond,'margin'),g(p,cond,'margin_ci'))
    if primary(r['rid'])=='32b' and 'correct' in p:
        row['coverage correct']=f1(g(p,'correct','coverage')); row['acc on claims made, correct']=f1(g(p,'correct','acc_made'))
    row['arb BA (other reader)']=f1(o.get('arb')); row['evid correct (other reader)']=f1(g(o,'correct','acc'))
    rows.append(row)
T_runs=pd.DataFrame(rows); print(T_runs.to_string(index=False))
T_runs.to_csv(OUT/'runs.csv', index=False); print('->', OUT/'runs.csv')

In [ ]:
#@title Mean over seeds
def msd(vals):
    vals=[v for v in vals if isinstance(v,(int,float)) and v==v]
    if not vals: return '–'
    return f'{st.mean(vals):.1f}' if len(vals)==1 else f'{st.mean(vals):.1f} ± {st.stdev(vals):.1f}'
by=defaultdict(list)
for r in RUNS: by[(r['dataset'],r['tag'],r['modality'],r['ratio'])].append(P(r['rid']))
rows=[]
for (ds,tag,mod,rt),ms in sorted(by.items()):
    rows.append({'dataset':ds,'model':tag,'input':mod,'rho':rt/100,'seeds':len(ms),
                 'arb BA':msd([m.get('arb') for m in ms]),
                 'evid correct':msd([g(m,'correct','acc') for m in ms]),'evid incorrect':msd([g(m,'contrastive','acc') for m in ms]),
                 'evid blind':msd([g(m,'blind','acc') for m in ms]),'evid substituted':msd([g(m,'substituted','acc') for m in ms]),
                 'margin correct':msd([g(m,'correct','margin') for m in ms]),'margin incorrect':msd([g(m,'contrastive','margin') for m in ms])})
T_seeds=pd.DataFrame(rows); print(T_seeds.to_string(index=False))
T_seeds.to_csv(OUT/'by_ratio.csv', index=False); print('->', OUT/'by_ratio.csv')

In [ ]:
#@title Reader agreement
def kappa(a,b):
    a=[str(x) for x in a]; b=[str(x) for x in b]; n=len(a)
    if not n: return float('nan'),float('nan')
    po=sum(x==y for x,y in zip(a,b))/n; ca,cb=Counter(a),Counter(b)
    pe=sum((ca[c]/n)*(cb[c]/n) for c in set(a)|set(b))
    return ((po-pe)/(1-pe) if pe<1 else float('nan')), po
def agreement(sel):
    rows=[]
    for q in QN:
        A=[];B=[]
        for (rid,cond),d in DATA.items():
            if not sel(RUN[rid]) or (rid,cond) not in JUD: continue
            J=JUD[(rid,cond)]
            for i,x in enumerate(d):
                if x['claims'][q] is None or i not in J: continue
                A.append(x['claims'][q]); B.append(J[i].get(q))
        k,po=kappa(A,B); rows.append({'field':QSHORT[q],'n':len(A),'agree %':round(100*po,1),'kappa':round(k,3)})
    for field in ('class','rejection'):
        A=[];B=[]
        for (rid,cond),d in DATA.items():
            if cond not in ('correct','contrastive') or not sel(RUN[rid]) or (rid,cond) not in JUD: continue
            J=JUD[(rid,cond)]
            for i,x in enumerate(d):
                if i not in J: continue
                if field=='class':
                    if x['class'] is None: continue
                    A.append(x['class']); B.append(J[i].get('class'))
                else:
                    if J[i].get('verdict') not in ('AGREE','DISAGREE','UNCLEAR'): continue
                    A.append(x['rejected']); B.append(J[i].get('verdict')=='DISAGREE')
        k,po=kappa(A,B); rows.append({'field':field,'n':len(A),'agree %':round(100*po,1),'kappa':round(k,3)})
    return pd.DataFrame(rows)
T_agree=agreement(lambda r: r['tag'] in PARSER_MODELS)
print(T_agree.to_string(index=False))
T_agree.to_csv(OUT/'reader_agreement.csv', index=False); print('->', OUT/'reader_agreement.csv')